必要なパッケージの読み込み、使用モデル等の指定

In [ ]:
from tqdm import tqdm
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import torch

modelname = 'rinna/japanese-gpt2-medium'
#modelname = 'rinna/japanese-gpt-1b'
num_epochs = 3
#num_epochs = 5
shortcut = False

データセットの読み込み

In [3]:
import pandas as pd
df_train = pd.read_csv('amazon_review/amazon_review_train2.csv')
df_test = pd.read_csv('amazon_review/amazon_review_test.csv')
df_valid = pd.read_csv('amazon_review/amazon_review_valid2.csv')
print(len(df_train),len(df_test),len(df_valid))
if shortcut:
    df_train = df_train[:15000]
    df_test = df_test[:560]
    df_valid = df_valid[:3700]
    print(len(df_train),len(df_test),len(df_valid))
df_train

150022 5654 37506


,Unnamed: 0.1,Unnamed: 0,sentence,label,review_id,j_lists,combined_text
0,0,120505,セイコー5のパーツは日本国内で生産されています。しかし、メカニカル時計の精度と品質に最も影響...,positive,R2EU75USYU74VW,"['セイコー', 'パーツ', '日本', '国内', '生産', 'する', 'れる', ...",セイコーパーツ日本国内生産するれるいるメカニカル時計精度品質影響及ぼす組み立て工程中国韓国マ...
1,1,62611,過去に様々な映像やドキュメンタリーになり、語り尽くされた有名な話ですが、エンターテインメント...,positive,RSGXU4S7LGNU8,"['過去', '様々', '映像', 'ドキュメンタリー', 'なる', '語る', '尽く...",過去様々映像ドキュメンタリーなる語る尽くすれる有名話エンターテインメント作品飽きることない最...
2,2,60879,すらすら読めておもしろい。あったかいエピソードがいろいろあって楽しめた。いろいろ編み物のすば...,positive,R23UVV73BVDFC2,"['読める', 'おもしろい', 'あったかい', 'エピソード', 'ある', '楽しめる...",読めるおもしろいあったかいエピソードある楽しめる編み物すばらしさ説くいる編み物やる思う編み物...
3,3,15749,面白かったので買ってみました。店のサンプルはもう少し難易度が低い物でしたがこれはうちの家族に...,positive,R39SIE6LJHTC5D,"['面白い', '買う', 'みる', '店', 'サンプル', '難易', '度', '低...",面白い買うみる店サンプル難易度低い物これうち家族難しい苦笑クリアする思う######面白かっ...
4,4,24426,機能が分かりやすく、使いやすいです!また、画面も綺麗で使用することが楽しいです!,positive,R19P9AEXV58G68,"['機能', '分かる', 'やすい', '使う', 'やすい', '画面', '綺麗', ...",機能分かるやすい使うやすい画面綺麗使用すること楽しい######機能が分かりやすく、使いやす...
...,...,...,...,...,...,...,...
150017,150017,119879,色彩豊かな絵と簡潔な文章で、想像力をかきたてられます。生後7ヶ月から子供に読み聞かせてます。...,positive,R2ZGGD4ET3PSTW,"['色彩', '豊か', '絵', '簡潔', '文章', '想像', '力', 'かきたて...",色彩豊か絵簡潔文章想像力かきたてるられる生後ヶ月子供読む聞くせるシンプル外装異なるカラフル動...
150018,150018,103694,今まで高機能な時計ばかり好んで使ってきましたが、よくよく考えると時刻を知る以外不要と言う事に...,positive,R1WW62ZW6C57J8,"['今', '機能', '時計', '好む', '使う', 'くる', '考える', '時刻...",今機能時計好む使うくる考える時刻知る以外不要言う事気が付く購入するみる見る見るレトロちるくる...
150019,150019,131932,水きりしやすいのが◎いろんなものを中に入れて沈めて遊んでいます。お風呂のおもちゃにはとても良...,positive,R3VL4XV3Q7P3RF,"['水きり', 'する', 'やすい', 'の', 'もの', '中', '入れる', '沈...",水きりするやすいのもの中入れる沈める遊ぶいる風呂おもちゃ良いよう######水きりしやすいの...
150020,150020,146867,一箱にたくさん入っていて、豪華です!5歳になったばかりの息子も夢中で組み立てて遊べたので大満...,positive,R1P5YI0IKGHBUF,"['一', '箱', 'たくさん', '入る', 'いる', '豪華', '歳', 'なる'...",一箱たくさん入るいる豪華歳なる息子夢中組み立てる遊べる満足######一箱にたくさん入ってい...


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

#df_train = pd.read_csv('amazon_review/amazon_review_train_valid.csv')
#df_test = pd.read_csv('amazon_review/amazon_review_test.csv')

tokenizer = AutoTokenizer.from_pretrained(modelname, use_fast=False)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [5]:
# モデルとトークナイザーの読み込み
tokenizer = AutoTokenizer.from_pretrained(modelname, use_fast=False)
tokenizer.do_lower_case = True
model = AutoModelForCausalLM.from_pretrained(modelname)

if False:  # 不要なのでスキップ
    tokenizer.add_special_tokens({'additional_special_tokens': ['######']})
    model.resize_token_embeddings(len(tokenizer))

if torch.cuda.is_available():
    if torch.cuda.device_count() > 1:
        device = torch.device("cuda:0")
        print('device = cuda:0')
    else:
        device = torch.device("cuda")
        print('device = cuda')
else:
    device = torch.device("cpu")
    print('device = cpu')
model.to(device)

# Optimizerの定義
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

device = cuda:0


In [6]:
ADD_BOS_TOKEN = True  # 文頭に<s>トークンを追加するかどうか

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, intexts, outtexts, transform=None, target_transform=None):
        self.data = []
        self.labels = []
        if len(intexts) != len(outtexts):
            raise ValueError("Input texts and output texts must have the same length.")
        for i in tqdm(range(len(intexts))):
            # 自立語系列
            inputs = tokenizer(intexts[i], add_special_tokens=False)
            if ADD_BOS_TOKEN:
                inputs['input_ids'] = [tokenizer.bos_token_id] + inputs['input_ids']
                inputs['attention_mask'] = [1] + inputs['attention_mask']
            inputs['input_ids'] = inputs['input_ids'] + [tokenizer.sep_token_id]
            inputs['attention_mask'] = inputs['attention_mask'] + [1]
            self.data.append(inputs.copy())
            inputs['input_ids'] = inputs['input_ids'][1:]
            inputs['attention_mask'] = inputs['attention_mask'][1:]
            self.labels.append(inputs.copy())
            # 完成された文
            inputs = tokenizer(outtexts[i], add_special_tokens=True)
            self.data[-1]['input_ids'] += inputs['input_ids'][:-1]
            self.data[-1]['attention_mask'] += inputs['attention_mask'][:-1]
            self.labels[-1]['input_ids'] += inputs['input_ids']
            self.labels[-1]['attention_mask'] += inputs['attention_mask']
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        d = self.data[idx]
        if self.transform:
            d = self.transform(d)
        l = self.labels[idx]
        if self.target_transform:
            l = self.target_transform(l)
        return d, l

train_dataset = TextDataset(df_train['j_lists'].apply(lambda x: ''.join(x[2:-2].split("', '"))).tolist(), df_train['sentence'].tolist())
print('train_dataset:', len(train_dataset))
valid_dataset = TextDataset(df_valid['j_lists'].apply(lambda x: ''.join(x[2:-2].split("', '"))).tolist(), df_valid['sentence'].tolist())
print('valid_dataset:', len(valid_dataset))
test_dataset = TextDataset(df_test['j_lists_test'].apply(lambda x: ''.join(x[2:-2].split("', '"))).tolist(), df_test['sentence'].tolist())
print('test_dataset:', len(test_dataset))

100%|██████████| 150022/150022 [00:58<00:00, 2577.04it/s]


train_dataset: 150022


100%|██████████| 37506/37506 [00:14<00:00, 2663.54it/s]


valid_dataset: 37506


100%|██████████| 5654/5654 [00:01<00:00, 2871.55it/s]

test_dataset: 5654


In [7]:
print(train_dataset[0][0])
print(train_dataset[0][1])
for i in range(len(train_dataset[0][0]['input_ids'])):
    print(f"{i:03d}: {train_dataset[0][0]['input_ids'][i]:5d}  {tokenizer.convert_ids_to_tokens(train_dataset[0][0]['input_ids'][i])}    |   {train_dataset[0][1]['input_ids'][i]:5d}  {tokenizer.convert_ids_to_tokens(train_dataset[0][1]['input_ids'][i])}")

{'input_ids': [1, 9, 3043, 1206, 7917, 5951, 795, 35, 2991, 546, 12720, 296, 769, 5303, 9276, 6693, 2332, 20016, 263, 13510, 10921, 496, 1305, 8172, 8568, 35, 2991, 546, 10, 8584, 96, 648, 600, 7917, 1200, 795, 13510, 1355, 1261, 13510, 10921, 13688, 56, 96, 905, 2264, 9681, 2341, 27720, 10, 23295, 1261, 546, 1079, 2341, 35, 3628, 96, 648, 23068, 35, 546, 693, 96, 648, 7506, 546, 9276, 9681, 264, 56, 1355, 1318, 9681, 9327, 35, 3923, 5086, 301, 6882, 2388, 9276, 1398, 12720, 296, 769, 11788, 256, 27719, 56, 538, 8596, 4870, 1791, 402, 535, 195, 9158, 838, 8453, 33, 1078, 474, 57, 900, 8404, 5, 9, 3043, 1206, 41, 10, 7917, 11, 9866, 795, 79, 20033, 8, 9680, 7, 12720, 296, 769, 5303, 10, 9276, 20, 6693, 17, 1016, 24687, 13510, 10921, 11103, 7, 1305, 7, 8172, 27554, 68, 10, 12, 8584, 2767, 8, 96, 648, 11, 81, 7034, 739, 41, 320, 25, 2767, 12, 7, 17774, 7, 7917, 11, 1200, 795, 19, 7, 13510, 11, 1355, 159, 3418, 8, 9798, 7, 13510, 10921, 11385, 7, 18408, 6277, 10, 2235, 9681, 10, 2341, 1561

In [ ]:
batch_size = 8

print('For Teacher Forcing Training')
def collate_fnTF(batch):
    data_list = {'input_ids': [], 'attention_mask': []}
    label_list = {'input_ids': [], 'attention_mask': []}
    # ミニバッチの中のデータ(d)とラベル(l)を一つずつ取り出してリストに追加
    for d, l in batch:
        data_list['input_ids'].append(torch.tensor(d['input_ids'], dtype=torch.int))
        data_list['attention_mask'].append(torch.tensor(d['attention_mask'], dtype=torch.int))
        label_list['input_ids'].append(torch.tensor(l['input_ids'], dtype=torch.long))
        label_list['attention_mask'].append(torch.tensor(l['attention_mask'], dtype=torch.long))
    # [pad]を詰めて長さを揃える
    src_batch = {'input_ids': torch.nn.utils.rnn.pad_sequence(data_list['input_ids'], batch_first=True, padding_value=tokenizer.pad_token_id, padding_side='right'), 
                 'attention_mask': torch.nn.utils.rnn.pad_sequence(data_list['attention_mask'], batch_first=True, padding_value=0)}
    tgt_batch = {'input_ids': torch.nn.utils.rnn.pad_sequence(label_list['input_ids'], batch_first=True, padding_value=tokenizer.pad_token_id, padding_side='right'), 
                 'attention_mask': torch.nn.utils.rnn.pad_sequence(label_list['attention_mask'], batch_first=True, padding_value=0)}
    return src_batch, tgt_batch

train_dataloaderTF = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fnTF, shuffle=True)
print('Train:', len(train_dataset), 'samples /', len(train_dataloaderTF), 'batches')
valid_dataloaderTF = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, collate_fn=collate_fnTF, shuffle=False)
print('Valid:', len(valid_dataset), 'samples /', len(valid_dataloaderTF), 'batches')
test_dataloaderTF = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fnTF, shuffle=False)
print('Test: ', len(test_dataset), 'samples /', len(test_dataloaderTF), 'batches')

For Teacher Forcing Training
Train: 150022 samples / 37506 batches
Valid: 37506 samples / 9377 batches
Test:  5654 samples / 1414 batches


In [9]:
print('For Non-teacher Forcing Training')
def collate_fnNTF(batch):
    data_list = {'input_ids': [], 'attention_mask': []}
    label_list = {'input_ids': [], 'attention_mask': []}
    # ミニバッチの中のデータ(d)とラベル(l)を一つずつ取り出してリストに追加
    for d, l in batch:
        print(d['input_ids'], tokenizer.convert_ids_to_tokens(d['input_ids']))
        i = d['input_ids'].index(tokenizer.sep_token_id) + 1
        d['input_ids'] = d['input_ids'][:i]
        d['attention_mask'] = d['attention_mask'][:i]
        data_list['input_ids'].append(torch.tensor(d['input_ids'], dtype=torch.int))
        data_list['attention_mask'].append(torch.tensor(d['attention_mask'], dtype=torch.int))
        print(l['input_ids'], tokenizer.convert_ids_to_tokens(l['input_ids']))
        i = l['input_ids'].index(tokenizer.sep_token_id) + 1
        l['input_ids'] = l['input_ids'][i:]
        l['attention_mask'] = l['attention_mask'][i:]
        label_list['input_ids'].append(torch.tensor(l['input_ids'], dtype=torch.long))
        label_list['attention_mask'].append(torch.tensor(l['attention_mask'], dtype=torch.long))
    # [pad]を詰めて長さを揃える
    src_batch = {'input_ids': torch.nn.utils.rnn.pad_sequence(data_list['input_ids'], batch_first=True, padding_value=tokenizer.pad_token_id, padding_side='left'), 
                 'attention_mask': torch.nn.utils.rnn.pad_sequence(data_list['attention_mask'], batch_first=True, padding_value=0)}
    tgt_batch = {'input_ids': torch.nn.utils.rnn.pad_sequence(label_list['input_ids'], batch_first=True, padding_value=tokenizer.pad_token_id, padding_side='right'), 
                 'attention_mask': torch.nn.utils.rnn.pad_sequence(label_list['attention_mask'], batch_first=True, padding_value=0)}
    return src_batch, tgt_batch

train_dataloaderNTF = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fnNTF, shuffle=True)
print('Train:', len(train_dataset), 'samples /', len(train_dataloaderTF), 'batches')
valid_dataloaderNTF = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, collate_fn=collate_fnNTF, shuffle=False)
print('Valid:', len(valid_dataset), 'samples /', len(valid_dataloaderTF), 'batches')
test_dataloaderNTF = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fnNTF, shuffle=False)
print('Test: ', len(test_dataset), 'samples /', len(test_dataloaderTF), 'batches')

For Non-teacher Forcing Training
Train: 150022 samples / 37506 batches
Valid: 37506 samples / 9377 batches
Test:  5654 samples / 1414 batches


In [10]:
# Inference mode (非teacher forcing)で使う model.generate() の使用例
# Inference mode は未実装

for batch in train_dataloaderNTF:
    print(batch[0]['input_ids'].shape[1], batch[1]['input_ids'].shape[1])
    outputs = model.generate(batch[0]['input_ids'].to(device), attention_mask=batch[0]['attention_mask'].to(device), 
                             pad_token_id=tokenizer.pad_token_id, 
                             max_new_tokens=40, 
                             return_dict_in_generate=True, 
                             output_scores=True)
    break

for i in range(len(outputs.sequences)):
    for j in range(len(outputs.scores)):
        print(tokenizer.decode(outputs.scores[j][i].argmax().item(), skip_special_tokens=False), end=' ')
    print('')

[1, 9, 35, 6778, 842, 95, 27507, 10810, 8405, 338, 2901, 307, 546, 7735, 8998, 35, 27507, 1529, 17490, 35, 1195, 160, 24492, 4780, 7642, 387, 936, 3028, 27206, 5, 9, 1362, 1315, 32, 101, 611, 7, 6778, 842, 3905, 27507, 7, 6129, 11126, 8447, 10810, 8405, 338, 12621, 546, 5632, 10, 19, 7, 2504, 128, 17, 8998, 3899, 29, 8, 862, 27507, 632, 7, 1529, 17, 17490, 3899, 29, 8, 1195, 3928, 24492, 4780, 10, 7642, 387, 936, 12, 3028, 10, 27206, 2767, 8] ['<s>', '▁', 'する', 'ふく', 'よ', 'か', '音質', '鮮', '烈', 'さ', '失', 'う', 'いる', 'わけ', '満足', 'する', '音質', '演奏', '感動', 'する', '個人', '的', 'ロッシ', 'ーニ', '序', '曲', '集', '一番', 'お気に入り', '[SEP]', '▁', 'ど', 'っ', 'し', 'り', 'とした', '、', 'ふく', 'よ', 'かな', '音質', '、', 'だから', 'といって', '決して', '鮮', '烈', 'さ', 'を失って', 'いる', 'わけではない', 'の', 'で', '、', 'おお', 'い', 'に', '満足', 'しま', 'した', '。', 'そして', '音質', 'よりも', '、', '演奏', 'に', '感動', 'しま', 'した', '。', '個人', '的には', 'ロッシ', 'ーニ', 'の', '序', '曲', '集', 'が', '一番', 'の', 'お気に入り', 'です', '。']
[9, 35, 6778, 842, 95, 27507, 10810, 8405, 338, 2901, 3

In [ ]:
teacher_forcing_ratio = 0.5


# CE損失
criterionCE = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# 微分可能なBLEU-1近似損失
# 離散BLEUはargmax/離散n-gram一致を使うため逆伝播できないので、
# 学習時は参照トークンへの予測確率から期待unigram precisionを計算する。
def criterionBLEU(logits, labels):
    # logits: [batch, vocab, sequence], labels: [batch, sequence]
    probabilities = torch.softmax(logits, dim=1).transpose(1, 2)  # [batch, sequence, vocab]
    valid_mask = labels.ne(tokenizer.pad_token_id)
    batch_scores = []

    for sample_probabilities, sample_labels, sample_mask in zip(
        probabilities, labels, valid_mask
    ):
        reference = sample_labels[sample_mask]
        if reference.numel() == 0:
            continue

        candidate_probabilities = sample_probabilities[sample_mask]
        reference_counts = torch.bincount(
            reference, minlength=probabilities.shape[-1]
        ).to(logits.dtype)
        expected_counts = candidate_probabilities.sum(dim=0)
        clipped_matches = torch.minimum(expected_counts, reference_counts).sum()
        precision = clipped_matches / reference.numel()

        # Teacher forcingでは候補長と参照長が同じだが、可変長入力にも対応する。
        candidate_length = candidate_probabilities.shape[0]
        brevity_exponent = torch.tensor(
            1.0 - reference.numel() / candidate_length,
            device=logits.device,
            dtype=logits.dtype,
        )
        brevity_penalty = torch.minimum(
            torch.ones((), device=logits.device, dtype=logits.dtype),
            torch.exp(brevity_exponent),
        )
        batch_scores.append(brevity_penalty * precision)

    if not batch_scores:
        return logits.new_zeros(())
    return torch.stack(batch_scores).mean()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    # epochに応じたalphaの切り替え
    if epoch < 4:
        alpha = 0.0
    elif epoch == 4:
        alpha = 0.001
    else:  # epoch == 5
        alpha = 0.002

    print(f"[Epoch {epoch}] alpha = {alpha}")

    for step, batch in enumerate(tqdm(train_dataloaderTF, desc="Train")):
        use_teacher_forcing = True if np.random.rand() < teacher_forcing_ratio else False
        inputs = batch[0]['input_ids'].to(device)
        attention_mask = batch[0]['attention_mask'].to(device)
        labels = batch[1]['input_ids'].to(device)
        optimizer.zero_grad()

        if use_teacher_forcing:
            outputs = model(input_ids=inputs, attention_mask=attention_mask)
            ce_loss = criterionCE(torch.transpose(outputs.logits, 1, 2), labels)
            bleu_reward = criterionBLEU(torch.transpose(outputs.logits, 1, 2), labels)
            loss = ce_loss + alpha * (1.0 - bleu_reward)
        else:
            # TODO: inference modeを実装したら、ここでmodel.generate()の出力に対する処理を行う。
            outputs = model(input_ids=inputs, attention_mask=attention_mask)
            ce_loss = criterionCE(torch.transpose(outputs.logits, 1, 2), labels)
            bleu_reward = criterionBLEU(torch.transpose(outputs.logits, 1, 2), labels)
            loss = ce_loss + alpha * (1.0 - bleu_reward)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_dataloaderTF)

    # 検証ループ
    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for batch in tqdm(valid_dataloaderTF, desc="Valid"):
            inputs = batch[0]['input_ids'].to(device)
            attention_mask = batch[0]['attention_mask'].to(device)
            labels = batch[1]['input_ids'].to(device)
            outputs = model(input_ids=inputs, attention_mask=attention_mask)
            ce_loss = criterionCE(torch.transpose(outputs.logits, 1, 2), labels)
            bleu_reward = criterionBLEU(torch.transpose(outputs.logits, 1, 2), labels)
            loss = ce_loss + alpha * (1.0 - bleu_reward)
            eval_loss += loss.item()
    avg_eval_loss = eval_loss / len(valid_dataloaderTF)

    print(f"Epoch {epoch}: Avg Train Loss = {avg_train_loss:.4f} / Avg Eval Loss: {avg_eval_loss:.4f}")

# 保存
save_d = './finetuned_model_1'
print("最終保存を実行")
model.save_pretrained(save_d)
tokenizer.save_pretrained(save_d)
torch.save(optimizer.state_dict(), f"{save_d}/optimizer.pt")
with open(f"{save_d}/meta.txt", "w") as f:
    f.write(f"epoch={epoch}\n")
    f.write(f"step={step}\n")
    f.write(f"loss={loss.item():.4f}\n")

[Epoch 0] alpha = 0.0


Valid: 100%|██████████| 9377/9377 [13:12<00:00, 11.83it/s]


Epoch 0: Avg Train Loss = 2.6794 / Avg Eval Loss: 2.5106
[Epoch 1] alpha = 0.0


Train:  82%|████████▏ | 30635/37506 [1:48:52<24:25,  4.69it/s]  


KeyboardInterrupt: 

: 

In [ ]:
generated_list = [""] * len(df_test)

pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

for i in range(len(df_test)):
    input_text = ''.join(df_test.loc[i, 'j_lists_test'][2:-2].split("', '"))
    if False:
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    else:
        inputs = tokenizer(input_text)
        if ADD_BOS_TOKEN:
            inputs['input_ids'] = [tokenizer.bos_token_id] + inputs['input_ids']
            inputs['attention_mask'] = [1] + inputs['attention_mask']
        inputs['input_ids'] = torch.tensor(inputs['input_ids'] + [tokenizer.sep_token_id], dtype=torch.int)[None, :]
        inputs['attention_mask'] = torch.tensor(inputs['attention_mask'] + [1], dtype=torch.int)[None, :]

    #length設定
    min_length = inputs["input_ids"].shape[1] +5
    max_length = inputs["input_ids"].shape[1] *2
    
    output = model.generate(
        input_ids=inputs["input_ids"].to(device),
        attention_mask=inputs["attention_mask"].to(device),
        min_length=min_length,
        max_length=max_length,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=1.2,
        num_return_sequences=1,
        pad_token_id=pad_token_id,
        #bos_token_id=tokenizer.bos_token_id,
        #eos_token_id=tokenizer.eos_token_id,
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    generated_list[i] = generated_text

    if i % 100 == 0:
        print(f"{i}件目:\n{input_text}\n{generated_text}")

In [ ]:
# 生成文と正解文のBLEU / BERTScore評価
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from bert_score import score as bertscore_score

references = df_test['sentence'].astype(str).tolist()
hypotheses = [str(text) for text in generated_list]

if len(references) != len(hypotheses):
    raise ValueError(
        f'生成文と正解文の件数が一致しません: {len(hypotheses)} != {len(references)}'
    )

# 日本語では空白区切りがないため、tokenizerのサブワード列でBLEUを計算する。
reference_tokens = [[tokenizer.tokenize(text)] for text in references]
hypothesis_tokens = [tokenizer.tokenize(text) for text in hypotheses]
bleu = corpus_bleu(
    reference_tokens,
    hypothesis_tokens,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=SmoothingFunction().method3,
)
print(f'BLEU-4: {bleu:.6f}')

# 日本語BERTを使ったBERTScore。初回実行時はモデルをダウンロードする。
bert_model_type = 'cl-tohoku/bert-base-japanese-v3'
precision, recall, f1 = bertscore_score(
    hypotheses,
    references,
    model_type=bert_model_type,
    lang='ja',
    device=str(device),
    verbose=True,
)
print(f'BERTScore Precision: {precision.mean().item():.6f}')
print(f'BERTScore Recall:    {recall.mean().item():.6f}')
print(f'BERTScore F1:        {f1.mean().item():.6f}')
